In [7]:
import os
import numpy as np
from PIL import Image

### section a

In [8]:
def load_dataset(folder_path, img_size=(28, 28)):
    images, labels = [], []
    for fname in sorted(os.listdir(folder_path)):
        if not fname.lower().endswith('.png'):
            continue
        label = int(fname.split('_')[0])
        img = Image.open(os.path.join(folder_path, fname)).convert('L')
        img = img.resize(img_size)
        arr = np.array(img, dtype=np.float32) / 255.0
        images.append(arr.flatten())
        labels.append(label)
    return np.array(images), np.array(labels)

### section b

In [9]:
def compute_prototypes(X, y):
    classes = np.unique(y)
    return {c: X[y == c].mean(axis=0) for c in classes}

def mdc_predict(X, prototypes):
    P = np.array(list(prototypes.values()))
    classes = list(prototypes.keys())
    dists = (X**2).sum(1, keepdims=True) + (P**2).sum(1) - 2 * X @ P.T
    return np.array([classes[i] for i in dists.argmin(axis=1)])

### section c

In [10]:
def classification_error(y_true, y_pred):
    return (y_true != y_pred).mean()

### section d

In [11]:
for group in ['a', 'b', 'c']:
    X_train, y_train = load_dataset(f'dataset/{group}/train')
    X_test,  y_test  = load_dataset(f'dataset/{group}/test')
    
    prototypes = compute_prototypes(X_train, y_train)
    
    train_err = classification_error(y_train, mdc_predict(X_train, prototypes))
    test_err  = classification_error(y_test,  mdc_predict(X_test,  prototypes))
    
    print(f'Group {group}: train_err={train_err:.2%}, test_err={test_err:.2%}')

Group a: train_err=0.00%, test_err=0.00%
Group b: train_err=1.25%, test_err=0.00%
Group c: train_err=1.00%, test_err=1.00%


### section e

<div dir="rtl" align="right">
با افزایش تعداد کلاس‌ها، خطای طبقه‌بندی افزایش می‌یابد؛ چون MDC هر کلاس را تنها با یک بردار میانگین نمایندگی می‌کند و با بیشتر شدن کلاس‌ها، فضای ویژگی شلوغ‌تر شده و پروتوتایپ‌های کلاس‌هایی که شکل ظاهری مشابهی دارند (مثل ۳ و ۸ یا ۱ و ۷) به هم نزدیک می‌شوند و باعث اشتباه در طبقه‌بندی می‌شوند.
</div>
